
# Ontological Unpacking of Business Intent into `rules.pl`

This notebook demonstrates how a natural-language business intent artifact is transformed into executable Prolog rules. The input is a business-oriented textual specification written as a marketing or retail analyst would write it. The output is a `rules.pl` file containing executable rules for:

1. semantic abstractions over products, brands, categories, purchases, and user interests;
2. unpacked semantic relations between products;
3. category discount prioritization; and
4. bundle discount candidate generation.


The Ontological Unpacking process implemented in this artifact is guided by concepts and modeling principles derived from the Unified Foundational Ontology (UFO), which serves as the ontological foundation that guides the transformation of informal business requirements into executable semantic structures. In particular, the unpacking process is inspired by UFO notions such as relators, material relations, roles, and truthmakers. These concepts help identify the semantic mechanisms that explain observable business relationships and support the construction of executable predicates. Consequently, the generated Prolog rules should be understood as ontology-guided abstractions rather than direct encodings of UFO constructs.

The LLM is used only during the knowledge-construction phase to propose candidate rules. The generated rules are then structurally validated and treated as a candidate executable model. The downstream recommendation notebook uses the resulting Prolog rules for symbolic inference.

## 1. Setup

This cell defines the input and output paths. The notebook expects `business_intent.txt` in the working directory. It writes the generated executable rules to `rules.pl`.

In [6]:
from pathlib import Path
import os
import re
import json
import textwrap
import subprocess
from typing import Dict, List, Tuple
from google.colab import drive

In [7]:
# Mount Google Drive when running in Google Colab.
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [8]:
# Input and output paths.
# Reviewers can change BASE_DIR if they run this notebook in a different environment.
BASE_DIR = Path("/content/drive/My Drive/KOA RecomSys/")

BUSINESS_INTENT_PATH = BASE_DIR / "business_intent.txt"
OUTPUT_RULES_PATH = BASE_DIR / "rules.pl"
LLM_RAW_OUTPUT_PATH = BASE_DIR / "rules_llm_raw_output.txt"
VALIDATION_REPORT_PATH = BASE_DIR / "rules_validation_report.json"

# Colab fallback: if files are mounted under /content or /mnt/data.
for candidate in [
    Path("/content/business_intent.txt"),
    Path("/mnt/data/business_intent.txt"),
]:
    if not BUSINESS_INTENT_PATH.exists() and candidate.exists():
        BUSINESS_INTENT_PATH = candidate

print("Business intent file:", BUSINESS_INTENT_PATH)
print("Output rules file:", OUTPUT_RULES_PATH)

Business intent file: /content/drive/My Drive/KOA RecomSys/business_intent.txt
Output rules file: /content/drive/My Drive/KOA RecomSys/rules.pl


## 2. Read the business intent artifact

This is the natural-language input produced from a business perspective. It is intentionally not written in ontology, Prolog, or LLM terminology.

In [9]:
def read_text(path: Path) -> str:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    return path.read_text(encoding="utf-8")

business_intent = read_text(BUSINESS_INTENT_PATH)

print("Business intent characters:", len(business_intent))
print("\nPreview:\n")
print(business_intent[:1200])

Business intent characters: 3282

Preview:

The company wants to improve the effectiveness of promotional campaigns by identifying product categories that are more likely to attract customer attention when discounts are offered. Categories should not be prioritized randomly or solely based on overall sales volume. Instead, the prioritization process should take into account signals that indicate a customer's actual interest in a category.

The strongest signal is when a customer explicitly expresses interest in a category, whether through search behavior, preferences, requests, or any other direct interaction. Categories explicitly mentioned by a customer should generally receive higher priority because they represent an active and intentional interest.

Another important signal is the relationship between products previously purchased by the customer and products that are frequently purchased together with them. If products from a particular category repeatedly appear as natural compl

## 3. Build the Ontological Unpacking prompt

The prompt constrains the LLM to generate Prolog rules over the predicates expected from the Semantic Onboarding stage. It also specifies that the output must contain executable Prolog only.

In [10]:
UNPACKING_PROMPT = """
You are supporting an Ontological Unpacking step for an enterprise decision-support experiment.

Your task is to transform the natural-language business intent below into a Prolog file named rules.pl.

IMPORTANT ROLE SEPARATION:
- The business analyst provides only the natural-language intent.
- You may propose candidate logical rules.
- You must not invent data facts.
- You must only use predicates expected to exist in the onboarded fact base.
- A domain analyst will later validate the generated rules before execution.

AVAILABLE FACT PREDICATES:
- functional_complex(ProductID, ProductName, Description, BrandID, CategoryID).
- agent(UserID, UserName, Attributes).
- buys(UserID, ProductID).
- also_buy(ProductID1, ProductID2).
- also_view(ProductID1, ProductID2).
- mentions(UserID, CategoryID).

EXPECTED OUTPUT:
Return only valid Prolog code, with no Markdown fences and no explanatory text outside comments.

REQUIRED RULE GROUPS:

1. Semantic abstractions
Create helper predicates that expose product name, brand, and category from functional_complex/5.
Create predicates for same_brand/2, same_category/2, related_also_buy/2, and related_also_view/2.
These predicates should be symmetric where appropriate and avoid recommending the same product as both sides of a relation.

2. Ontological unpacking
Create unpack_relation/3 predicates that materialize the business meaning of product relations.
The rules should capture at least:
- Ecosystem Synergy: products connected by also_buy and belonging to the same brand.
- Solution Composition: products connected by also_buy and belonging to different categories.
- Quality Competition: products connected by also_view and belonging to the same category.
Also create a predicate that aligns a product with a user's expressed category interest.

3. Semantic recommendations
Create semantic_recommendation/3 and explain_recommendation/3 predicates.
The recommendation must be derived from explicit symbolic predicates, not from text similarity or implicit language-model inference.

4. Category discount prioritization
Create weights for the following evidence types:
- explicit category mention;
- also-buy targets in the category;
- previous purchases in the category.
Create predicates to count previous purchases and also-buy targets by category.
Create prioritized_discount_category/7 with the following arguments:
UserID, CatID, PriorityScore, MentionFlag, BoughtCount, AlsoBuyCount, Explanation.
The score must combine the three evidence types as a weighted sum.
A category should be a candidate only when there is behavioral evidence from prior purchases or also-buy opportunities.

5. Bundle discount candidates
Create weights for:
- also_buy;
- also_view;
- same_brand;
- same_category.
Create candidate_bundle_pair/2, bundle_score/3, bundle_explanation/3, and bundle_discount_candidate/6.
The bundle score must be computed from explicit semantic relations only.

STYLE REQUIREMENTS:
- Use clear English comments.
- Use readable predicate names.
- Do not include Portuguese comments.
- Do not include placeholder facts.
- Do not include example data.
- Do not include Markdown.

BUSINESS INTENT:
\"\"\"
{business_intent}
\"\"\"
"""

prompt = UNPACKING_PROMPT.format(business_intent=business_intent)

print("Prompt characters:", len(prompt))
print(prompt[:2500])

Prompt characters: 6457

You are supporting an Ontological Unpacking step for an enterprise decision-support experiment.

Your task is to transform the natural-language business intent below into a Prolog file named rules.pl.

IMPORTANT ROLE SEPARATION:
- The business analyst provides only the natural-language intent.
- You may propose candidate logical rules.
- You must not invent data facts.
- You must only use predicates expected to exist in the onboarded fact base.
- A domain analyst will later validate the generated rules before execution.

AVAILABLE FACT PREDICATES:
- functional_complex(ProductID, ProductName, Description, BrandID, CategoryID).
- agent(UserID, UserName, Attributes).
- buys(UserID, ProductID).
- also_buy(ProductID1, ProductID2).
- also_view(ProductID1, ProductID2).
- mentions(UserID, CategoryID).

EXPECTED OUTPUT:
Return only valid Prolog code, with no Markdown fences and no explanatory text outside comments.

REQUIRED RULE GROUPS:

1. Semantic abstractions
Create

## 4. Optional LLM-based rule generation

Run this cell if you want to call Gemini and regenerate `rules.pl` from the business intent artifact. If no API key is available, skip this cell and use the validated reference fallback in the next section.

In [11]:
def strip_markdown_fences(text: str) -> str:
    """Remove Markdown code fences if the LLM returns them despite the instruction."""
    text = text.strip()
    text = re.sub(r"^\s*```(?:prolog|pl)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```\s*$", "", text)
    return text.strip()

def generate_rules_with_gemini(prompt: str, model_name: str = "gemini-2.0-flash") -> str:
    """
    Generate rules.pl using Gemini.

    The API key must be available in one of these environment variables:
    - GEMINI_API_KEY
    - GOOGLE_API_KEY

    In Google Colab, reviewers may set it through:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    """
    api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    if not api_key:
        raise RuntimeError(
            "No Gemini API key found. Set GEMINI_API_KEY or GOOGLE_API_KEY, "
            "or use the validated reference fallback cell below."
        )

    try:
        from google import genai
    except ImportError as exc:
        raise ImportError(
            "Package google-genai is not installed. Run: pip install google-genai"
        ) from exc

    client = genai.Client(api_key=api_key)
    response = client.models.generate_content(
        model=model_name,
        contents=prompt,
    )
    return strip_markdown_fences(response.text)

# Uncomment to generate rules.pl with Gemini.
# generated_rules = generate_rules_with_gemini(prompt)
# LLM_RAW_OUTPUT_PATH.write_text(generated_rules, encoding="utf-8")
# OUTPUT_RULES_PATH.write_text(generated_rules, encoding="utf-8")
# print("Generated rules written to:", OUTPUT_RULES_PATH)


## 5. Validated reference output

For reproducibility, this notebook includes the domain-validated `rules.pl` used in the experiment. This allows reviewers to inspect and reproduce the executable artifact without depending on an external API call.

In [12]:
VALIDATED_REFERENCE_RULES = r"""% ==========================================================
% RULES.PL - Semantic Bridging Logic for Recommendations
% ==========================================================

% ----------------------------------------------------------
% 1. SEMANTIC ABSTRACTIONS
% ----------------------------------------------------------

product_name(P, Name) :-
    functional_complex(P, Name, _, _, _).

product_brand(P, BrandID) :-
    functional_complex(P, _, _, BrandID, _).

product_category(P, CatID) :-
    functional_complex(P, _, _, _, CatID).

same_brand(P1, P2) :-
    product_brand(P1, BrandID),
    product_brand(P2, BrandID),
    P1 \= P2.

same_category(P1, P2) :-
    product_category(P1, CatID),
    product_category(P2, CatID),
    P1 \= P2.

related_also_buy(P1, P2) :-
    also_buy(P1, P2) ;
    also_buy(P2, P1).

related_also_view(P1, P2) :-
    also_view(P1, P2) ;
    also_view(P2, P1).

% ----------------------------------------------------------
% 2. ONTOLOGICAL UNPACKING
% ----------------------------------------------------------

unpack_relation(P1, P2, 'Ecosystem Synergy') :-
    related_also_buy(P1, P2),
    same_brand(P1, P2).

unpack_relation(P1, P2, 'Solution Composition') :-
    related_also_buy(P1, P2),
    product_category(P1, Cat1),
    product_category(P2, Cat2),
    Cat1 \= Cat2.

unpack_relation(P1, P2, 'Quality Competition') :-
    related_also_view(P1, P2),
    same_category(P1, P2).

user_intent_alignment(UserID, ProdID) :-
    mentions(UserID, CatID),
    product_category(ProdID, CatID).

% ----------------------------------------------------------
% 3. SEMANTIC RECOMMENDATIONS
% ----------------------------------------------------------

semantic_recommendation(UserID, TargetProd, Reason) :-
    buys(UserID, BoughtProd),
    unpack_relation(BoughtProd, TargetProd, UnpackedType),
    \+ buys(UserID, TargetProd),
    user_intent_alignment(UserID, TargetProd),
    atomic_list_concat(
        ['This item offers ', UnpackedType,
         ' and aligns with your interest in this category.'],
        Reason
    ).

semantic_recommendation(UserID, TargetProd, 'Brand Loyalty') :-
    buys(UserID, PreviousProd),
    same_brand(PreviousProd, TargetProd),
    \+ buys(UserID, TargetProd),
    user_intent_alignment(UserID, TargetProd).

semantic_recommendation(UserID, TargetProd, Reason) :-
    buys(UserID, BoughtProd),
    unpack_relation(BoughtProd, TargetProd, UnpackedType),
    \+ buys(UserID, TargetProd),
    \+ user_intent_alignment(UserID, TargetProd),
    atomic_list_concat(
        ['Based on previous purchases, this provides ', UnpackedType, '.'],
        Reason
    ).

explain_recommendation(UserID, ProdID, Explanation) :-
    semantic_recommendation(UserID, ProdID, Reason),
    product_name(ProdID, Name),
    agent(UserID, UserName, _),
    atomic_list_concat(
        ['Hello ', UserName, ', we suggest ', Name, '. Reason: ', Reason],
        Explanation
    ).

% ----------------------------------------------------------
% 4. DISCOUNT PRIORITIZATION
% ----------------------------------------------------------

% Revised weights: buys and also_buy dominate the decision.
weight_discount(mention, 1).
weight_discount(also_buy_category, 6).
weight_discount(previous_purchase_category, 5).

count_user_purchases_in_category(UserID, CatID, Count) :-
    findall(ProdID,
        (
            buys(UserID, ProdID),
            product_category(ProdID, CatID)
        ),
        ProdList),
    sort(ProdList, UniqueProdList),
    length(UniqueProdList, Count).

count_user_also_buy_targets_in_category(UserID, CatID, Count) :-
    findall(TargetProd,
        (
            buys(UserID, BoughtProd),
            related_also_buy(BoughtProd, TargetProd),
            product_category(TargetProd, CatID),
            \+ buys(UserID, TargetProd)
        ),
        ProdList),
    sort(ProdList, UniqueProdList),
    length(UniqueProdList, Count).

user_mentions_category(UserID, CatID) :-
    mentions(UserID, CatID).

% Revised semantics:
% A category is a candidate only when real behavioral evidence is available.
candidate_discount_category(UserID, CatID, BoughtCount, AlsoBuyCount) :-
    count_user_purchases_in_category(UserID, CatID, BoughtCount),
    count_user_also_buy_targets_in_category(UserID, CatID, AlsoBuyCount),
    (BoughtCount > 0 ; AlsoBuyCount > 0).

% Single consistent predicate for CQ2.
prioritized_discount_category(UserID, CatID, PriorityScore, MentionFlag, BoughtCount, AlsoBuyCount, Explanation) :-
    candidate_discount_category(UserID, CatID, BoughtCount, AlsoBuyCount),
    weight_discount(mention, MentionWeight),
    weight_discount(also_buy_category, AlsoBuyWeight),
    weight_discount(previous_purchase_category, BoughtWeight),
    ( user_mentions_category(UserID, CatID) ->
        MentionFlag = yes,
        MentionScore = MentionWeight
    ;
        MentionFlag = no,
        MentionScore = 0
    ),
    PriorityScore is MentionScore + (AlsoBuyCount * AlsoBuyWeight) + (BoughtCount * BoughtWeight),
    PriorityScore > 0,
    atomic_list_concat([
        'Category ', CatID,
        ' prioritized because mention=', MentionFlag,
        ', also_buy_candidates=', AlsoBuyCount,
        ', previous_purchases=', BoughtCount
    ], Explanation).

% ----------------------------------------------------------
% 5. BUNDLE DISCOUNT CANDIDATES
% ----------------------------------------------------------

weight_bundle(also_buy, 5).
weight_bundle(also_view, 3).
weight_bundle(same_brand, 2).
weight_bundle(same_category, 1).

candidate_bundle_pair(P1, P2) :-
    P1 @< P2,
    (
        related_also_buy(P1, P2) ;
        related_also_view(P1, P2) ;
        same_brand(P1, P2) ;
        same_category(P1, P2)
    ).

bundle_score(P1, P2, Score) :-
    weight_bundle(also_buy, WAlsoBuy),
    weight_bundle(also_view, WAlsoView),
    weight_bundle(same_brand, WBrand),
    weight_bundle(same_category, WCategory),
    ( related_also_buy(P1, P2) -> AlsoBuyScore = WAlsoBuy ; AlsoBuyScore = 0 ),
    ( related_also_view(P1, P2) -> AlsoViewScore = WAlsoView ; AlsoViewScore = 0 ),
    ( same_brand(P1, P2) -> BrandScore = WBrand ; BrandScore = 0 ),
    ( same_category(P1, P2) -> CategoryScore = WCategory ; CategoryScore = 0 ),
    Score is AlsoBuyScore + AlsoViewScore + BrandScore + CategoryScore.

bundle_explanation(P1, P2, Explanation) :-
    ( related_also_buy(P1, P2) -> AlsoBuyText = 'yes' ; AlsoBuyText = 'no' ),
    ( related_also_view(P1, P2) -> AlsoViewText = 'yes' ; AlsoViewText = 'no' ),
    ( same_brand(P1, P2) -> BrandText = 'yes' ; BrandText = 'no' ),
    ( same_category(P1, P2) -> CategoryText = 'yes' ; CategoryText = 'no' ),
    atomic_list_concat([
        'Bundle candidate because also_buy=', AlsoBuyText,
        ', also_view=', AlsoViewText,
        ', same_brand=', BrandText,
        ', same_category=', CategoryText
    ], Explanation).

bundle_discount_candidate(ProductID1, ProductName1, ProductID2, ProductName2, BundleScore, Explanation) :-
    candidate_bundle_pair(ProductID1, ProductID2),
    bundle_score(ProductID1, ProductID2, BundleScore),
    BundleScore > 0,
    product_name(ProductID1, ProductName1),
    product_name(ProductID2, ProductName2),
    bundle_explanation(ProductID1, ProductID2, Explanation)."""

# This fallback writes the domain-validated rules used in the experiment.
# It is useful for reviewers who want to reproduce the artifact without calling an external LLM.
OUTPUT_RULES_PATH.write_text(VALIDATED_REFERENCE_RULES, encoding="utf-8")
print("Validated rules.pl written to:", OUTPUT_RULES_PATH)
print("Characters:", len(VALIDATED_REFERENCE_RULES))


Validated rules.pl written to: /content/drive/My Drive/KOA RecomSys/rules.pl
Characters: 7250


## 6. Structural validation

The generated rules are checked for required predicates, accidental Markdown fences, placeholder input facts, and remaining Portuguese markers in comments or code.

In [13]:
REQUIRED_PREDICATES = [
    "product_name",
    "product_brand",
    "product_category",
    "same_brand",
    "same_category",
    "related_also_buy",
    "related_also_view",
    "unpack_relation",
    "user_intent_alignment",
    "semantic_recommendation",
    "explain_recommendation",
    "weight_discount",
    "count_user_purchases_in_category",
    "count_user_also_buy_targets_in_category",
    "candidate_discount_category",
    "prioritized_discount_category",
    "weight_bundle",
    "candidate_bundle_pair",
    "bundle_score",
    "bundle_explanation",
    "bundle_discount_candidate",
]

PORTUGUESE_MARKERS = [
    "categoria", "candidato", "predicado", "semântica", "decisão",
    "compras", "usuário", "produto", "evidência", "priorização"
]

def validate_rules_text(rules_text: str) -> Dict[str, object]:
    """Run lightweight structural checks over generated Prolog code."""
    present = {}
    for predicate in REQUIRED_PREDICATES:
        pattern = rf"(^|\n)\s*{re.escape(predicate)}\s*\("
        present[predicate] = bool(re.search(pattern, rules_text))

    missing = [predicate for predicate, ok in present.items() if not ok]

    portuguese_found = sorted({
        marker for marker in PORTUGUESE_MARKERS
        if re.search(rf"\b{re.escape(marker)}\b", rules_text, flags=re.IGNORECASE)
    })

    has_markdown = "```" in rules_text
    has_placeholder_facts = bool(re.search(r"(^|\n)\s*(functional_complex|agent|buys|also_buy|also_view|mentions)\s*\(", rules_text))

    return {
        "required_predicates_present": present,
        "missing_required_predicates": missing,
        "portuguese_markers_found": portuguese_found,
        "contains_markdown_fences": has_markdown,
        "contains_placeholder_input_facts": has_placeholder_facts,
        "passed": not missing and not portuguese_found and not has_markdown and not has_placeholder_facts,
    }

rules_text = OUTPUT_RULES_PATH.read_text(encoding="utf-8")
validation_report = validate_rules_text(rules_text)
VALIDATION_REPORT_PATH.write_text(json.dumps(validation_report, indent=2), encoding="utf-8")

print(json.dumps(validation_report, indent=2))


{
  "required_predicates_present": {
    "product_name": true,
    "product_brand": true,
    "product_category": true,
    "same_brand": true,
    "same_category": true,
    "related_also_buy": true,
    "related_also_view": true,
    "unpack_relation": true,
    "user_intent_alignment": true,
    "semantic_recommendation": true,
    "explain_recommendation": true,
    "weight_discount": true,
    "count_user_purchases_in_category": true,
    "count_user_also_buy_targets_in_category": true,
    "candidate_discount_category": true,
    "prioritized_discount_category": true,
    "weight_bundle": true,
    "candidate_bundle_pair": true,
    "bundle_score": true,
    "bundle_explanation": true,
    "bundle_discount_candidate": true
  },
  "missing_required_predicates": [],
  "portuguese_markers_found": [],
  "contains_markdown_fences": false,
  "contains_placeholder_input_facts": true,
  "passed": false
}


## 7. Inspect the generated rule sections

This cell prints the main sections of `rules.pl` so that reviewers can inspect how business intent was transformed into executable predicates.

In [14]:
def extract_rules_by_section(rules_text: str) -> Dict[str, str]:
    """Split rules.pl into commented sections to help reviewers inspect the artifact."""
    sections = {}
    current = "Preamble"
    buffer = []

    for line in rules_text.splitlines():
        section_match = re.match(r"%\s*\d+\.\s*(.+)", line.strip())
        if section_match:
            sections[current] = "\n".join(buffer).strip()
            current = section_match.group(1).strip()
            buffer = [line]
        else:
            buffer.append(line)

    sections[current] = "\n".join(buffer).strip()
    return {k: v for k, v in sections.items() if v}

sections = extract_rules_by_section(rules_text)

for name, content in sections.items():
    print("=" * 80)
    print(name)
    print("=" * 80)
    print(content[:1600])
    print()


Preamble
% ==========================================================
% RULES.PL - Semantic Bridging Logic for Recommendations
% ==========================================================

% ----------------------------------------------------------

SEMANTIC ABSTRACTIONS
% 1. SEMANTIC ABSTRACTIONS
% ----------------------------------------------------------

product_name(P, Name) :-
    functional_complex(P, Name, _, _, _).

product_brand(P, BrandID) :-
    functional_complex(P, _, _, BrandID, _).

product_category(P, CatID) :-
    functional_complex(P, _, _, _, CatID).

same_brand(P1, P2) :-
    product_brand(P1, BrandID),
    product_brand(P2, BrandID),
    P1 \= P2.

same_category(P1, P2) :-
    product_category(P1, CatID),
    product_category(P2, CatID),
    P1 \= P2.

related_also_buy(P1, P2) :-
    also_buy(P1, P2) ;
    also_buy(P2, P1).

related_also_view(P1, P2) :-
    also_view(P1, P2) ;
    also_view(P2, P1).

% ----------------------------------------------------------

O

## 8. Reviewer-oriented explanation

This final note summarizes why this artifact supports the claim that Ontological Unpacking generates executable predicates that participate in the reasoning process.

In [15]:
reviewer_explanation = """
Reviewer note: how this artifact evidences Ontological Unpacking

The input file business_intent.txt is written as a business-facing requirement.
It does not mention Prolog, UFO, ontologies, or implementation predicates.

The generated rules.pl makes the unpacking step explicit:
- business notions such as brand ecosystem, complementary purchases, and bundle suitability
  are materialized as executable predicates;
- unpack_relation/3 represents the semantic interpretation of product relations;
- prioritized_discount_category/7 formalizes discount prioritization as a weighted decision rule;
- bundle_discount_candidate/6 derives bundle candidates from explicit symbolic relations;
- the resulting predicates are later used by the recommendation notebook for symbolic inference
  and logical soundness evaluation.

Thus, the ontological layer is not only descriptive. It becomes part of the executable rule base.
"""

print(reviewer_explanation)



Reviewer note: how this artifact evidences Ontological Unpacking

The input file business_intent.txt is written as a business-facing requirement.
It does not mention Prolog, UFO, ontologies, or implementation predicates.

The generated rules.pl makes the unpacking step explicit:
- business notions such as brand ecosystem, complementary purchases, and bundle suitability
  are materialized as executable predicates;
- unpack_relation/3 represents the semantic interpretation of product relations;
- prioritized_discount_category/7 formalizes discount prioritization as a weighted decision rule;
- bundle_discount_candidate/6 derives bundle candidates from explicit symbolic relations;
- the resulting predicates are later used by the recommendation notebook for symbolic inference
  and logical soundness evaluation.

Thus, the ontological layer is not only descriptive. It becomes part of the executable rule base.

